In [7]:
import os
import itertools
import pandas as pd
import numpy as np
import time

from core.ingest import load_stock_data
from core.prepare import prepare_pipeline
from core import svr
from core.analysis import run_analysis_pipeline
from core.graphics import run_graphics_pipeline
from core.promptSender import send_prompt_to_google, send_prompt_to_deepseek

CSV_PATH = "stock_prices_daily.csv"
LISTA_TICKERS = ["AAPL","ABBV","AMAT"]
LISTA_KERNELS = ["linear"]
LISTA_SPLITS = [0.8]
LISTA_C = [1.0, 10.0]
LISTA_WINDOW_RATIOS = [0.05, 0.10, 0.15]
EPSILON_LIST = [0.01, 0.1, 1.0]



def gerar_combinacoes_parametros(tickers, kernels, splits, c_values, window_ratios,epsilon_values):
    combinacoes = list(itertools.product(tickers, kernels, splits, c_values, window_ratios,epsilon_values))
    print(f"-> Planejamento concluído: {len(combinacoes)} cenários mapeados.")
    return combinacoes

def inicializar_estrutura_resultado(dicionario, ticker, kernel, split, c_value, w_ratio, epsilon_value):

    if ticker not in dicionario:
        dicionario[ticker] = {}
    if kernel not in dicionario[ticker]:
        dicionario[ticker][kernel] = {}
    if split not in dicionario[ticker][kernel]:
        dicionario[ticker][kernel][split] = {}
    if c_value not in dicionario[ticker][kernel][split]:
        dicionario[ticker][kernel][split][c_value] = {}
    if w_ratio not in dicionario[ticker][kernel][split][c_value]:
        dicionario[ticker][kernel][split][c_value][w_ratio] = {}
    if epsilon_value not in dicionario[ticker][kernel][split][c_value][w_ratio]:
        dicionario[ticker][kernel][split][c_value][w_ratio][epsilon_value] = {}

In [8]:
def llm_call(prompts_dict,ticker,respostas_llm,dados_preparados_llm):

        print(f"\n--- Enviando prompt (few-shot) do gemini para o ativo: {ticker} ---")
        resposta_gemini_few = send_prompt_to_google(prompts_dict["few-shot"])
        
        print(f"\n--- Enviando prompt (zero) do gemini para o ativo: {ticker} ---")
        resposta_gemini_zero = send_prompt_to_google(prompts_dict["zero-shot"])

        print(f"\n--- Enviando prompt para o DEEPSEEK (few-shot) - Ativo: {ticker} ---")
        resposta_deepseek_few = send_prompt_to_deepseek(prompts_dict["few-shot"])

        print(f"\n--- Enviando prompt para o DEEPSEEK (zero-shot) - Ativo: {ticker} ---")
        resposta_deepseek_zero = send_prompt_to_deepseek(prompts_dict["zero-shot"])
        
        respostas_llm[ticker] = {
            "resposta_few": {"gemini":resposta_gemini_few,"deepseek":resposta_deepseek_few},
            "resposta_zero":{"gemini":resposta_gemini_zero,"deepseek":resposta_deepseek_zero},
            "prompt_usado_few_shot": prompts_dict["few-shot"],
            "prompt_usado_zero_shot": prompts_dict["zero-shot"],
            "raw_llm_data": dados_preparados_llm[ticker]["llm"]
        }
        return respostas_llm

In [9]:
def main():

    try:
        df_completo = load_stock_data(tickers=LISTA_TICKERS, file_path=CSV_PATH)
    except FileNotFoundError as e:
        print(f"\n[Erro] Não foi possível carregar os dados: {e}")
        return

    respostas_llm = {}
    
    dados_preparados_llm = prepare_pipeline(
        df_completo, 
        split_ratio=LISTA_SPLITS[0], 
        window_ratio=LISTA_WINDOW_RATIOS[0], 
        target_col="Close",
        llm_days=30
    )
    
    for ticker in LISTA_TICKERS:
        if ticker in dados_preparados_llm:
            prompts_dict = dados_preparados_llm[ticker]["llm"]["prompts"]
            respostas_llm = llm_call(prompts_dict, ticker, respostas_llm, dados_preparados_llm)
        
    cenarios = gerar_combinacoes_parametros(
        LISTA_TICKERS, LISTA_KERNELS, LISTA_SPLITS, LISTA_C, LISTA_WINDOW_RATIOS, EPSILON_LIST
    )
    
    resultados_finais = {}
    cache_preparacao = {}

    for idx, (ticker, kernel, split, c_value, w_ratio, epsilon_value) in enumerate(cenarios, start=1):
        
        inicializar_estrutura_resultado(
            resultados_finais, ticker, kernel, split, c_value, w_ratio, epsilon_value
        )

        chave_prep = (split, w_ratio)

        if chave_prep not in cache_preparacao:
            cache_preparacao[chave_prep] = prepare_pipeline(
                df_completo, 
                split_ratio=split, 
                window_ratio=w_ratio, 
                target_col="Close"
            )
        
        dados_preparados = cache_preparacao[chave_prep]
        ativo_dict = dados_preparados[ticker]
        
        X_train, y_train, X_test, y_test, scaler = svr.unpack_active_data(ativo_dict)

        t_start_train = time.time()
        model = svr.train_svr(X_train, y_train, kernel=kernel, C=c_value, epsilon=epsilon_value)
        t_end_train = time.time()
        tempo_treino = t_end_train - t_start_train

        t_start_pred = time.time()
        predictions_scaled = svr.predict_svr(model, X_test)
        t_end_pred = time.time()
        tempo_predicao = t_end_pred - t_start_pred

        tempo_total = tempo_treino + tempo_predicao

        predictions_real = svr.denormalize_data(predictions_scaled, scaler)
        answers_real = svr.denormalize_data(y_test, scaler)
        
        resultados_finais[ticker][kernel][split][c_value][w_ratio][epsilon_value] = {
            "predicted": predictions_real,
            "answers": answers_real,
            "tempo_treino": tempo_treino,
            "tempo_predicao": tempo_predicao,
            "tempo_total": tempo_total
        }
        
    analise_campeoes = run_analysis_pipeline(resultados_finais)
    tabelas_geradas = run_graphics_pipeline(analise_campeoes, output_dir="resultados_tabelas")

main()


--- Enviando prompt (few-shot) do gemini para o ativo: AAPL ---
<pergunta>
Você irá prever o valor de uma ação.Você receberá conjuntos de 5 valores de ultimos dias  juntos de seus gabaritos para treinar e tentar adivinhar o próximo.

Exemplo 1: 273.50408935546875 , 272.82470703125 , 271.6058349609375 , 270.75665283203125 , 267.0101623535156 , Gabarito: 262.1147155761719
Exemplo 2: 260.08660888671875 , 258.7978515625 , 259.1275329589844 , 260.0067138671875 , 260.8059387207031 , Gabarito: 259.71697998046875
Exemplo 3: 257.9685974121094 , 255.29112243652344 , 246.4693756103516 , 247.41848754882807 , 248.1178436279297 , Gabarito: 247.80812072753903
Exemplo 4: 255.17123413085935 , 258.0285339355469 , 256.2002868652344 , 258.0385437011719 , 259.2374267578125 , Gabarito: 269.7575988769531
Exemplo 5: 269.22808837890625 , 276.23150634765625 , 275.6520690917969 , 277.8599853515625 , 274.6199951171875 , Gabarito: 273.67999267578125

Os valores são: 275.6520690917969 , 277.8599853515625 , 274.619